In [ ]:
# creating the data set for scatter plot
import pandas as pd

# load and clean column names
df = pd.read_csv("cleaned_rental_data.csv")
df.columns = df.columns.str.strip().str.replace(" ", "_")

# drop rows where Neighborhood is missing
df = df[df["Neighborhood"].notna()]

# fill missing values with neighborhood-specific averages
df["Personal_Income_Avg"] = df.groupby("Neighborhood")["Personal_Income_Avg"].transform(
    lambda x: x.fillna(x.mean())
)
df["Total_Rent_Avg"] = df.groupby("Neighborhood")["Total_Rent_Avg"].transform(
    lambda x: x.fillna(x.mean())
)

scatter_data = df[["Personal_Income_Avg", "Total_Rent_Avg", "Neighborhood"]].dropna()

scatter_data
# save for D3
# scatter_data.to_csv("income_vs_rent.csv", index=False)


,Personal_Income_Avg,Total_Rent_Avg,Neighborhood
0,5454.545455,3500.0,The Fenway/Kenmore
1,3000.000000,4500.0,Roxbury
2,4800.000000,4500.0,Roxbury
3,4800.000000,4500.0,Roxbury
4,2000.000000,3500.0,Mission Hill
...,...,...,...
85,9000.000000,3500.0,Seaport District
86,9000.000000,3500.0,Seaport District
87,3000.000000,4500.0,Roxbury
88,3000.000000,4500.0,Mission Hill


In [15]:
# creating an interactive map
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm  

# load the rental dataset
rental_data = pd.read_csv("cleaned_rental_data.csv")

# clean column names
rental_data.columns = (
    rental_data.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)

# define mapping to match GeoJSON neighborhood names
name_mapping = {
    "The_Fenway/Kenmore": "Fenway",
    "Mission_Hill": "Mission Hill",
    "Roxbury": "Roxbury",
    "Downtown": "Downtown",
    "BackBay": "Back Bay",
    "Beacon_Hill": "Beacon Hill",
    "North_End": "North End",
    "Waterfront": "Waterfront",
    "Jamaica_Plain_(JP)": "Jamaica Plain",
    "South_Boston": "South Boston",
    "West_End_(residential_area_near_the_North_End_and_TD_Garden)": "West End",
    "Financial_District_(located_near_Seaport_and_the_Waterfront,_known_for_its_office_buildings,_businesses,_and_luxury_condos.)": "Financial District",
    "Seaport_District": "South Boston Waterfront"
}

# clean neighborhood names and apply mapping
rental_data["Neighborhood"] = rental_data["Neighborhood"].str.strip().str.replace(" ", "_")
rental_data["Neighborhood_Clean"] = rental_data["Neighborhood"].map(name_mapping)
rental_data["Neighborhood_Clean"] = rental_data["Neighborhood_Clean"].fillna(
    rental_data["Neighborhood"].str.replace("_", " ")
)

# aggregate stats
summary = rental_data.groupby("Neighborhood_Clean").agg(
    Avg_Bedrooms=('Bedrooms', 'mean'),
    Avg_Bathrooms=('Bathrooms', 'mean'),
    Furnished_Count=('Furnished', lambda x: (x == 'Furnished').sum()),
    Partially_Furnished_Count=('Furnished', lambda x: (x == 'Partially').sum()),
    Unfurnished_Count=('Furnished', lambda x: (x == 'Unfurnished').sum()),
    Avg_Rent=('Total_Rent_Avg', 'mean'),
    Avg_Utility_Cost=('Utility_Cost_Avg', 'mean'),
    Avg_Income=('Personal_Income_Avg', 'mean'),
    Avg_Total_Cost=('Total_Rent_w__Utilities', 'mean'),
    Pet_Friendly=('Pet-friendly', 'sum'),
    Pool=('Pool', 'sum'),
    Game_Room=('Game_room', 'sum'),
    Free_Laundry=('Laundry_facilities_(no_need_to_pay)', 'sum'),
    Concierge=('Concierge_Doorman', 'sum'),
    Parking=('Parking', 'sum'),
    Roof_Deck=('roof_deck', 'sum'),
    Mail_Room=('Mail_Room', 'sum'),
    Gym=('Gym', 'sum')
).reset_index()

# add computed metrics
summary["Rent_to_Income_Ratio"] = summary["Avg_Rent"] / summary["Avg_Income"]
summary["Cost_Per_Person"] = summary["Avg_Rent"] / summary["Avg_Bedrooms"]
summary["Total_Cost"] = summary["Avg_Total_Cost"] / summary["Avg_Bedrooms"]

# create HTML popup
def create_popup(row):
    return f"""
    <b>Neighborhood:</b> {row['Neighborhood_Clean']}<br>
    <b>Avg Bedrooms:</b> {row['Avg_Bedrooms']:.2f}<br>
    <b>Avg Bathrooms:</b> {row['Avg_Bathrooms']:.2f}<br>
    <b>Furnished:</b> {row['Furnished_Count']} |
    <b>Partially:</b> {row['Partially_Furnished_Count']} |
    <b>Unfurnished:</b> {row['Unfurnished_Count']}<br>
    <b>Avg Rent:</b> ${row['Avg_Rent']:.2f}<br>
    <b>Avg Utility Cost:</b> ${row['Avg_Utility_Cost']:.2f}<br>
    <b>Avg Personal Income:</b> ${row['Avg_Income']:.2f}<br>
    <b>Rent-to-Income Ratio:</b> {row['Rent_to_Income_Ratio']:.2f}<br>
    <b>Cost per Person:</b> ${row['Cost_Per_Person']:.2f}<br>
    <b>Total Cost per Person:</b> ${row['Total_Cost']:.2f}<br><br>
    <b>Amenities Count:</b><br>
    Pet-friendly: {int(row['Pet_Friendly'])} | Pool: {int(row['Pool'])} | Game Room: {int(row['Game_Room'])}<br>
    Free Laundry: {int(row['Free_Laundry'])} | Concierge: {int(row['Concierge'])}<br>
    Parking: {int(row['Parking'])} | Roof Deck: {int(row['Roof_Deck'])}<br>
    Mail Room: {int(row['Mail_Room'])} | Gym: {int(row['Gym'])}
    """

summary["popup_html"] = summary.apply(create_popup, axis=1)

# load GeoJSON
geojson_path = "boston.geojson"
boston_geo = gpd.read_file(geojson_path)

# merge with summary
merged = boston_geo.merge(summary, left_on="name", right_on="Neighborhood_Clean", how="left")

# drop datetime columns
for col in merged.columns:
    if pd.api.types.is_datetime64_any_dtype(merged[col]):
        merged = merged.drop(columns=[col])

# convert popup to string
merged["popup_html"] = merged["popup_html"].astype(str)
# normalize fill color by Cost Per Person
cost_min = merged["Cost_Per_Person"].min()
cost_max = merged["Cost_Per_Person"].max()
color_scale = cm.linear.YlOrRd_09.scale(cost_min, cost_max)

# create Folium map
m = folium.Map(location=[42.3601, -71.0589], zoom_start=12, tiles="cartodbpositron")

folium.TileLayer(
    tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    name='Light OSM',
    attr='OpenStreetMap',
    opacity=0.2, 
).add_to(m)

def style_function(feature):
    cost = feature["properties"].get("Cost_Per_Person")
    ratio = feature["properties"].get("Rent_to_Income_Ratio")

    # case 1: ratio is NaN → black fill
    if ratio is None or pd.isna(ratio):
        return {
            'fillColor': 'black',
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.3
        }

    # case 2: ratio is valid (> 0) and cost is valid → use color scale
    if ratio > 0 and cost is not None and not pd.isna(cost):
        return {
            'fillColor': color_scale(cost),
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }

    # case 3: ratio is valid but cost is NaN → fallback (e.g. light yellow)
    return {
        'fillColor': '#fff9c4',  # light yellow
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.3
    }

# set up tooltip logic
def tooltip_function(feature):
    name = feature["properties"].get("name")
    cost = feature["properties"].get("Cost_Per_Person")
    if pd.isna(cost):
        return f"{name} — No data available"
    return f"{name}"

folium.GeoJson(
    merged,
    name="Boston Rent Map",
    tooltip=folium.GeoJsonTooltip(
        fields=["name"],
        aliases=["Neighborhood:"],
        labels=False,
        style=("background-color: white; color: #333; font-size: 12px; padding: 5px;")
    ),
    popup=folium.GeoJsonPopup(fields=["popup_html"], labels=False, sticky=False),
    style_function=style_function
).add_to(m)

# add legend to map
color_scale.caption = "Cost Per Person (Avg Rent ÷ Avg Bedrooms)"
color_scale.add_to(m)

m


#save html
output_path = "boston_rent_map.html"
m.save(output_path)
